In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch_geometric.loader import DataLoader as GeoDataLoader
from torch_geometric.nn import global_mean_pool
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader


c:\Users\Admin\Desktop\ai_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
import pickle
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import numpy as np
import pandas as pd
from numpy import savetxt, loadtxt
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.nn import Linear, BatchNorm1d
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import Input, Concatenate, Reshape, Conv1D, Flatten, Dense, Dropout, MultiHeadAttention, LayerNormalization, Add
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
from transformers import AutoTokenizer, AutoModel

from tqdm import tqdm
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, Dropout, Dense, concatenate, BatchNormalization



In [3]:
def load_dataset():

    data = pd.read_csv("DATASET/multilabel_BILSTM_BERT.csv", index_col=False)
    data = data.drop(columns='Unnamed: 0')
    print(len(data))

    label_columns = ['Other', 'access_control', 'arithmetic', 'denial_service',
                     'front_running', 'reentrancy', 'time_manipulation',
                     'unchecked_low_calls']

    for column in label_columns:
        data[column] = data[column].apply(lambda x: 1 if x != 0 else 0)

    # Giữ đúng 39645 rows
    #data = data.iloc[:39645]
    total_rows = len(data)
    split_point = int(0.8 * total_rows)

    print(split_point)
    print(total_rows - split_point)

    df_train = data.iloc[:split_point]
    df_test = data.iloc[split_point:]
    

    if True:
        X_train_source = np.load("DATASET/X_train_source_minmaxsummean_10chunks_512.npy")
        X_test_source = np.load("DATASET/X_test_source_minmaxsummean_10chunks_512.npy")
        X_train_source = X_train_source.reshape(-1, 10, 4, 768)
        X_test_source = X_test_source.reshape(-1, 10, 4, 768)
        X_train_source = X_train_source[:, :, 0, :]
        X_test_source = X_test_source[:, :, 0, :]
        X_train_source = X_train_source.reshape(-1, 10 * 768)
        X_test_source = X_test_source.reshape(-1, 10 * 768)

    else:

        # =========================
        # CODEBERT EMBEDDING
        # =========================

        source_train = df_train['sourcecode'].tolist()
        source_test = df_test['sourcecode'].tolist()
        

        tokenizer = AutoTokenizer.from_pretrained("./codebert")
        model = AutoModel.from_pretrained("./codebert")

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device)


        model.eval()

        print("Generating CodeBERT embeddings...")

        X_train_source = get_codebert_embeddings(source_train, tokenizer, model, device)
        X_test_source = get_codebert_embeddings(source_test, tokenizer, model, device)
        
            
        # texts = df_train['sourcecode'].astype(str).tolist()

        # lengths = [len(tokenizer.tokenize(x)) for x in texts]

        # print("Average tokens:", np.mean(lengths))
        # print("Max tokens:", np.max(lengths))


        print("Load Features CodeBERT success!!")
        np.save("DATASET/X_train_source.npy", X_train_source)
        np.save("DATASET/X_test_source.npy", X_test_source)

    # =========================
    # OPCODE FEATURES (giữ nguyên)
    # =========================

    X_train_opcode = np.array(df_train.iloc[:, 11:].values)
    X_test_opcode = np.array(df_test.iloc[:, 11:].values)

    # =========================
    # LABELS
    # =========================

    sl4 = label_columns[:4]
    sl5 = label_columns[:5]
    sl6 = label_columns[:6]
    sl7 = label_columns[:7]
    sl8 = label_columns[:8]

    y_train4 = df_train[sl4].values
    y_test4 = df_test[sl4].values
    y_train5 = df_train[sl5].values
    y_test5 = df_test[sl5].values
    y_train6 = df_train[sl6].values
    y_test6 = df_test[sl6].values
    y_train7 = df_train[sl7].values
    y_test7 = df_test[sl7].values
    y_train8 = df_train[sl8].values
    y_test8 = df_test[sl8].values
    
    labels = data[sl8].values

    print(f"Load Data for x-Label MultiLabel success!!")

    return X_train_opcode, X_test_opcode, X_train_source, X_test_source, y_train4, y_test4, y_train5, y_test5, y_train6, y_test6, y_train7, y_test7, y_train8, y_test8, labels

In [4]:
X_train_opcode, X_test_opcode, X_train_source, X_test_source, y_train4, y_test4, y_train5, y_test5, y_train6, y_test6, y_train7, y_test7, y_train8, y_test8, labels = load_dataset()

45597
36477
9120
Load Data for x-Label MultiLabel success!!


In [5]:
import torch
import pickle
import numpy as np

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

# =========================
# LOAD DATA
# =========================
input_file = 'DATASET/gnn_input.pkl'

with open(input_file, 'rb') as f:
    raw_data = pickle.load(f)

feature_list = raw_data['features']
adj_matrices = raw_data['adj_matrices']
label_list = raw_data['labels']

target_dim = 57

# =========================
# UTILS
# =========================
def pad_or_truncate_features(features, target_dim):
    if features.shape[1] > target_dim:
        return features[:, :target_dim]

    elif features.shape[1] < target_dim:
        padded_features = np.zeros(
            (features.shape[0], target_dim),
            dtype=np.float32
        )
        padded_features[:, :features.shape[1]] = features
        return padded_features

    else:
        return features.astype(np.float32)


def create_pyg_data(features, adj_matrix, label, target_dim):
    adj = torch.tensor(np.array(adj_matrix), dtype=torch.long)

    edge_index = adj.nonzero(as_tuple=False).t().contiguous()

    x = torch.tensor(
        pad_or_truncate_features(features, target_dim),
        dtype=torch.float
    )

    y = torch.tensor(label, dtype=torch.float)

    return Data(
        x=x,
        edge_index=edge_index,
        y=y
    )


# =========================
# CREATE 5 DATASETS
# LABEL SIZE = 4 -> 8
# =========================
datasets = {}

for num_labels in range(4, 9):

    data_list = []

    for feat, adj, label in zip(
        feature_list,
        adj_matrices,
        label_list
    ):

        # lấy num_labels label đầu tiên
        sub_label = label[:num_labels]

        data = create_pyg_data(
            feat,
            adj,
            sub_label,
            target_dim
        )

        data_list.append(data)

    # nếu muốn shuffle trước khi split
    # indices = np.random.permutation(len(data_list))
    # data_list = [data_list[i] for i in indices]

    split_idx = int(len(data_list) * 0.8)

    train_loader = DataLoader(
        data_list[:split_idx],
        batch_size=64,
        shuffle=False
    )

    test_loader = DataLoader(
        data_list[split_idx:],
        batch_size=64,
        shuffle=False
    )

    datasets[f'loader_{num_labels}'] = {
        'train_loader': train_loader,
        'test_loader': test_loader,
        'num_classes': num_labels
    }

# =========================
# ACCESS
# =========================

cfg_train_loader4 = datasets['loader_4']['train_loader']
cfg_test_loader4  = datasets['loader_4']['test_loader']

cfg_train_loader5 = datasets['loader_5']['train_loader']
cfg_test_loader5  = datasets['loader_5']['test_loader']

cfg_train_loader6 = datasets['loader_6']['train_loader']
cfg_test_loader6  = datasets['loader_6']['test_loader']

cfg_train_loader7 = datasets['loader_7']['train_loader']
cfg_test_loader7  = datasets['loader_7']['test_loader']

cfg_train_loader8 = datasets['loader_8']['train_loader']
cfg_test_loader8  = datasets['loader_8']['test_loader']

In [6]:

class OpcodeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)     # input cho Embedding
        self.y = torch.tensor(y, dtype=torch.float32)  # cho BCELoss

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
class SourceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [7]:
# =========================
# OPCODE DATASETS
# =========================

opcode_train_dataset4 = OpcodeDataset(X_train_opcode, y_train4)
opcode_test_dataset4  = OpcodeDataset(X_test_opcode, y_test4)

opcode_train_dataset5 = OpcodeDataset(X_train_opcode, y_train5)
opcode_test_dataset5  = OpcodeDataset(X_test_opcode, y_test5)

opcode_train_dataset6 = OpcodeDataset(X_train_opcode, y_train6)
opcode_test_dataset6  = OpcodeDataset(X_test_opcode, y_test6)

opcode_train_dataset7 = OpcodeDataset(X_train_opcode, y_train7)
opcode_test_dataset7  = OpcodeDataset(X_test_opcode, y_test7)

opcode_train_dataset8 = OpcodeDataset(X_train_opcode, y_train8)
opcode_test_dataset8  = OpcodeDataset(X_test_opcode, y_test8)


# =========================
# OPCODE LOADERS
# =========================

opcode_train_loader4 = DataLoader(opcode_train_dataset4, batch_size=64, shuffle=False)
opcode_test_loader4  = DataLoader(opcode_test_dataset4, batch_size=64, shuffle=False)

opcode_train_loader5 = DataLoader(opcode_train_dataset5, batch_size=64, shuffle=False)
opcode_test_loader5  = DataLoader(opcode_test_dataset5, batch_size=64, shuffle=False)

opcode_train_loader6 = DataLoader(opcode_train_dataset6, batch_size=64, shuffle=False)
opcode_test_loader6  = DataLoader(opcode_test_dataset6, batch_size=64, shuffle=False)

opcode_train_loader7 = DataLoader(opcode_train_dataset7, batch_size=64, shuffle=False)
opcode_test_loader7  = DataLoader(opcode_test_dataset7, batch_size=64, shuffle=False)

opcode_train_loader8 = DataLoader(opcode_train_dataset8, batch_size=64, shuffle=False)
opcode_test_loader8  = DataLoader(opcode_test_dataset8, batch_size=64, shuffle=False)


# =========================
# SOURCE DATASETS
# =========================

source_train_dataset4 = SourceDataset(X_train_source, y_train4)
source_test_dataset4  = SourceDataset(X_test_source, y_test4)

source_train_dataset5 = SourceDataset(X_train_source, y_train5)
source_test_dataset5  = SourceDataset(X_test_source, y_test5)

source_train_dataset6 = SourceDataset(X_train_source, y_train6)
source_test_dataset6  = SourceDataset(X_test_source, y_test6)

source_train_dataset7 = SourceDataset(X_train_source, y_train7)
source_test_dataset7  = SourceDataset(X_test_source, y_test7)

source_train_dataset8 = SourceDataset(X_train_source, y_train8)
source_test_dataset8  = SourceDataset(X_test_source, y_test8)


# =========================
# SOURCE LOADERS
# =========================

source_train_loader4 = DataLoader(source_train_dataset4, batch_size=64, shuffle=False)
source_test_loader4  = DataLoader(source_test_dataset4, batch_size=64, shuffle=False)

source_train_loader5 = DataLoader(source_train_dataset5, batch_size=64, shuffle=False)
source_test_loader5  = DataLoader(source_test_dataset5, batch_size=64, shuffle=False)

source_train_loader6 = DataLoader(source_train_dataset6, batch_size=64, shuffle=False)
source_test_loader6  = DataLoader(source_test_dataset6, batch_size=64, shuffle=False)

source_train_loader7 = DataLoader(source_train_dataset7, batch_size=64, shuffle=False)
source_test_loader7  = DataLoader(source_test_dataset7, batch_size=64, shuffle=False)

source_train_loader8 = DataLoader(source_train_dataset8, batch_size=64, shuffle=False)
source_test_loader8  = DataLoader(source_test_dataset8, batch_size=64, shuffle=False)

In [8]:

class BiLSTMModel(nn.Module):
    def __init__(
        self,
        vocab_size=34000,
        embedding_dim=256,
        hidden_dim=128,
        output_dim=8
    ):
        super(BiLSTMModel, self).__init__()
        
        # Embedding
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Stacked BiLSTM (2 layers)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=2,              # 🔥 2 layer BiLSTM
            batch_first=True,
            bidirectional=True,
            dropout=0.3               # dropout giữa các layer LSTM
        )
        
        # Regularization
        self.dropout = nn.Dropout(0.3)
        self.bn1 = nn.BatchNorm1d(hidden_dim * 2)
        
        # Dense layers
        self.fc1 = nn.Linear(hidden_dim * 2, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.fc2 = nn.Linear(128, 64)
        
        # Output
        self.out = nn.Linear(64, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (B, 280)
        x = self.embedding(x)  # -> (B, 280, 286)
        
        # LSTM
        _, (h_n, _) = self.lstm(x)
        
        # Lấy hidden state cuối của 2 chiều (forward + backward)
        x = torch.cat((h_n[-2], h_n[-1]), dim=1)  # (B, 256)
        
        # Dense
        x = self.dropout(x)
        x = self.bn1(x)
        x = self.dropout(x)
        
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.bn2(x)
        x = self.dropout(x)
        
        x = self.fc2(x)
        x = torch.relu(x)
        return x
        

import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool

class GAT(nn.Module):
    def __init__(
        self,
        in_channels=57,
        hidden_channels=64,
        num_classes=8
    ):
        super(GAT, self).__init__()

        # ===== GAT Layers =====
        self.conv1 = GATConv(
            in_channels=in_channels,
            out_channels=hidden_channels,
            heads=4,
            dropout=0.3
        )

        self.conv2 = GATConv(
            in_channels=hidden_channels * 4,
            out_channels=hidden_channels,
            heads=2,
            dropout=0.3
        )

        # ===== Batch Normalization =====
        self.bn1 = nn.BatchNorm1d(hidden_channels * 4)
        self.bn2 = nn.BatchNorm1d(hidden_channels * 2)

        # ===== Dense Layers =====
        self.fc1 = nn.Linear(hidden_channels * 2, hidden_channels)

        # ===== Output Layer =====
        self.fc2 = nn.Linear(hidden_channels, num_classes)

        self.dropout = 0.3

    def forward(self, data):

        # ===== Inputs =====
        x = data.x
        edge_index = data.edge_index
        batch = data.batch

        # ===== GAT Layer 1 =====
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        # ===== GAT Layer 2 =====
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        # ===== Global Pooling =====
        x = global_mean_pool(x, batch)

        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        # ===== Dense Layer =====
        x = self.fc1(x)
        x = F.relu(x)
        return x

In [9]:
codebert_model = load_model('DATASET/bert.h5')
codebert_model = Model(
    inputs=codebert_model.input,
    outputs=codebert_model.layers[-2].output   # layer 64
)

bilstm_model = BiLSTMModel()
bilstm_model.load_state_dict(torch.load("DATASET/bilstm_model.pth"))
device = "cuda" if torch.cuda.is_available() else "cpu"
bilstm_model.to(device)
bilstm_model.eval()

gat_model = GAT()
gat_model.load_state_dict(torch.load("DATASET/gat.pth"))
gat_model.to(device)
gat_model.eval()

C:\Users\Admin\AppData\Local\Temp\ipykernel_15096\1889956571.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  bilstm_model.load_state_dict(torch.load("DATASET/bilstm_mode

GAT(
  (conv1): GATConv(57, 64, heads=4)
  (conv2): GATConv(256, 64, heads=2)
  (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=8, bias=True)
)

In [10]:
for i, layer in enumerate(codebert_model.layers):
    try:
        print(i, layer.name, layer.output_shape)
    except:
        print(i, layer.name, "no shape")

0 sourcecode_features no shape
1 batch_normalization_3 no shape
2 dense_6 no shape
3 dropout_4 no shape
4 dense_7 no shape
5 batch_normalization_4 no shape
6 dropout_5 no shape
7 dense_8 no shape
8 dropout_6 no shape
9 dense_9 no shape
10 batch_normalization_5 no shape
11 dropout_7 no shape
12 dense_10 no shape


In [11]:
# class MetaClassifier(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(192, 128),
#             nn.ReLU(),
#             nn.Dropout(0.3),

#             nn.Linear(128, 64),
#             nn.ReLU(),

#             nn.Linear(64, 8),
#             nn.Sigmoid()
#         )

#     def forward(self, x):
#         return self.net(x)

In [12]:
# class MetaClassifier(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.net = nn.Sequential(

#             nn.Linear(192, 8),
#             nn.Sigmoid()
#         )

#     def forward(self, x):
#         return self.net(x)
    
import torch
import torch.nn as nn


class MetaClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(

            # ===== Fusion Layer 1 =====
            nn.Linear(192, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            # ===== Fusion Layer 2 =====
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            # ===== Output =====
            nn.Linear(64, 4),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

# import torch
# import torch.nn as nn


# class MetaClassifier(nn.Module):
#     def __init__(self):
#         super().__init__()

#         # ===== Conv1D Fusion =====
#         self.conv1d = nn.Conv1d(
#             in_channels=1,
#             out_channels=64,
#             kernel_size=3
#         )

#         # ===== Pooling =====
#         self.global_pool = nn.AdaptiveMaxPool1d(1)

#         # ===== Regularization =====
#         self.dropout = nn.Dropout(0.3)

#         # ===== Dense Layers =====
#         self.fc1 = nn.Linear(64, 64)

#         # ===== Output =====
#         self.out = nn.Linear(64, 8)

#     def forward(self, x):

#         # x shape:
#         # (batch_size, 192)

#         # ===== Reshape for Conv1D =====
#         x = x.unsqueeze(1)

#         # (batch, 1, 192)

#         # ===== Conv1D =====
#         x = self.conv1d(x)

#         # (batch, 64, 190)

#         x = torch.relu(x)

#         # ===== Global Max Pooling =====
#         x = self.global_pool(x)

#         # (batch, 64, 1)

#         x = x.squeeze(-1)

#         # (batch, 64)

#         # ===== Dropout =====
#         x = self.dropout(x)

#         # ===== Dense Layer =====
#         x = self.fc1(x)

#         x = torch.relu(x)

#         # ===== Output =====
#         x = self.out(x)

#         x = torch.sigmoid(x)

#         return x

In [13]:
def extract_opcode_features(model, loader, device):
    model.eval()
    outputs = []

    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            out = model(x)   # (B, 8)
            outputs.append(out.cpu())

    return torch.cat(outputs, dim=0)  # (N, 8)


def extract_gat_features(model, loader, device):
    model.eval()
    outputs = []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data)   # (B, 8)
            outputs.append(out.cpu())

    return torch.cat(outputs, dim=0)


codebert_features_train = codebert_model.predict(X_train_source, batch_size=64)
codebert_features_test  = codebert_model.predict(X_test_source, batch_size=64)

opcode_feat_train = extract_opcode_features(bilstm_model, opcode_train_loader4, device)
opcode_feat_test  = extract_opcode_features(bilstm_model, opcode_test_loader4, device)

gat_feat_train = extract_gat_features(gat_model, cfg_train_loader4, device)
gat_feat_test  = extract_gat_features(gat_model, cfg_test_loader4, device)

# CodeBERT (Keras)
codebert_feat_train = codebert_model.predict(X_train_source, batch_size=64)
codebert_feat_test  = codebert_model.predict(X_test_source, batch_size=64)

# convert numpy -> torch nếu cần
codebert_feat_train = torch.tensor(codebert_feat_train)
codebert_feat_test  = torch.tensor(codebert_feat_test)

train_features = torch.cat([
    opcode_feat_train,
    gat_feat_train,
    codebert_feat_train
], dim=1)   # (N, 24)

test_features = torch.cat([
    opcode_feat_test,
    gat_feat_test,
    codebert_feat_test
], dim=1)

570/570 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step
143/143 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
570/570 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step
143/143 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step


In [14]:
class FusionDataset(Dataset):
    def __init__(self, X, y):
        self.X = X.float()
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [15]:
train_dataset4 = FusionDataset(train_features, y_train4)
test_dataset4  = FusionDataset(test_features, y_test4)

train_loader4 = DataLoader(train_dataset4, batch_size=64, shuffle=True)
test_loader4  = DataLoader(test_dataset4, batch_size=64, shuffle=False)

train_dataset5 = FusionDataset(train_features, y_train5)
test_dataset5  = FusionDataset(test_features, y_test5)

train_loader5 = DataLoader(train_dataset5, batch_size=64, shuffle=True)
test_loader5  = DataLoader(test_dataset5, batch_size=64, shuffle=False)


train_dataset6 = FusionDataset(train_features, y_train6)
test_dataset6  = FusionDataset(test_features, y_test6)

train_loader6 = DataLoader(train_dataset6, batch_size=64, shuffle=True)
test_loader6  = DataLoader(test_dataset6, batch_size=64, shuffle=False)


train_dataset7 = FusionDataset(train_features, y_train7)
test_dataset7  = FusionDataset(test_features, y_test7)

train_loader7 = DataLoader(train_dataset7, batch_size=64, shuffle=True)
test_loader7  = DataLoader(test_dataset7, batch_size=64, shuffle=False)


train_dataset8 = FusionDataset(train_features, y_train8)
test_dataset8  = FusionDataset(test_features, y_test8)

train_loader8 = DataLoader(train_dataset8, batch_size=64, shuffle=True)
test_loader8  = DataLoader(test_dataset8, batch_size=64, shuffle=False)

In [16]:
import copy
import torch
import torch.nn as nn
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    hamming_loss
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =========================================================
# MODEL
# =========================================================

class MetaClassifier(nn.Module):

    def __init__(self, num_labels=4):
        super().__init__()

        self.feature_extractor = nn.Sequential(

            # ===== Fusion Layer 1 =====
            nn.Linear(192, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            # ===== Fusion Layer 2 =====
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        self.classifier = nn.Linear(64, num_labels)

    def forward(self, x):

        x = self.feature_extractor(x)

        logits = self.classifier(x)

        return logits


# =========================================================
# EXPAND OUTPUT LAYER
# =========================================================

def expand_classifier(old_model, new_num_labels):

    old_num_labels = old_model.classifier.out_features

    new_model = MetaClassifier(
        num_labels=new_num_labels
    )

    # ================= COPY BACKBONE =================
    new_model.feature_extractor.load_state_dict(
        old_model.feature_extractor.state_dict()
    )

    # ================= COPY OLD HEAD =================
    with torch.no_grad():

        new_model.classifier.weight[:old_num_labels] = \
            old_model.classifier.weight

        new_model.classifier.bias[:old_num_labels] = \
            old_model.classifier.bias

    return new_model


# =========================================================
# FREEZE BACKBONE
# =========================================================

def freeze_backbone(model):

    for param in model.feature_extractor.parameters():
        param.requires_grad = False


# =========================================================
# UNFREEZE
# =========================================================

def unfreeze_all(model):

    for param in model.parameters():
        param.requires_grad = True


# =========================================================
# TRAIN
# =========================================================

def train_model(
        model,
        train_loader,
        test_loader,
        epochs=100,
        lr=1e-3
):

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr
    )

    model.to(device)

    best_model = None
    best_f1 = 0

    for epoch in range(epochs):

        # ================= TRAIN =================
        model.train()

        total_loss = 0

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            logits = model(X_batch)

            loss = criterion(logits, y_batch)

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        # ================= EVALUATE =================
        metrics = evaluate_model(
            model,
            test_loader
        )

        # print(
        #     f"Epoch [{epoch+1}/{epochs}] | "
        #     f"Loss: {avg_loss:.4f} | "
        #     f"F1-Micro: {metrics['f1_micro']:.4f} | "
        #     f"F1-Macro: {metrics['f1_macro']:.4f}"
        # )

        if metrics['f1_micro'] > best_f1:

            best_f1 = metrics['f1_micro']

            best_model = copy.deepcopy(model)

    return best_model


# =========================================================
# EVALUATION
# =========================================================
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)

import torch


def evaluate_model(
        model,
        data_loader,
        device='cuda',
        threshold=0.5
):

    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():

        for X_batch, y_batch in data_loader:

            X_batch = X_batch.to(device)

            logits = model(X_batch)

            probs = torch.sigmoid(logits)

            preds = (probs > threshold).float()

            y_true.extend(
                y_batch.cpu().numpy()
            )

            y_pred.extend(
                preds.cpu().numpy()
            )

    # =====================================================
    # F1 SCORE
    # =====================================================

    f1_micro = f1_score(
        y_true,
        y_pred,
        average='micro',
        zero_division=0
    )

    f1_macro = f1_score(
        y_true,
        y_pred,
        average='macro',
        zero_division=0
    )

    f1_weighted = f1_score(
        y_true,
        y_pred,
        average='weighted',
        zero_division=0
    )

    f1_samples = f1_score(
        y_true,
        y_pred,
        average='samples',
        zero_division=0
    )

    # =====================================================
    # PRECISION
    # =====================================================

    precision_micro = precision_score(
        y_true,
        y_pred,
        average='micro',
        zero_division=0
    )

    precision_macro = precision_score(
        y_true,
        y_pred,
        average='macro',
        zero_division=0
    )

    precision_weighted = precision_score(
        y_true,
        y_pred,
        average='weighted',
        zero_division=0
    )

    precision_samples = precision_score(
        y_true,
        y_pred,
        average='samples',
        zero_division=0
    )

    # =====================================================
    # RECALL
    # =====================================================

    recall_micro = recall_score(
        y_true,
        y_pred,
        average='micro',
        zero_division=0
    )

    recall_macro = recall_score(
        y_true,
        y_pred,
        average='macro',
        zero_division=0
    )

    recall_weighted = recall_score(
        y_true,
        y_pred,
        average='weighted',
        zero_division=0
    )

    recall_samples = recall_score(
        y_true,
        y_pred,
        average='samples',
        zero_division=0
    )

    # =====================================================
    # RETURN
    # =====================================================

    return {

        # ================= F1 =================
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "f1_samples": f1_samples,

        # ================= PRECISION =================
        "precision_micro": precision_micro,
        "precision_macro": precision_macro,
        "precision_weighted": precision_weighted,
        "precision_samples": precision_samples,

        # ================= RECALL =================
        "recall_micro": recall_micro,
        "recall_macro": recall_macro,
        "recall_weighted": recall_weighted,
        "recall_samples": recall_samples,
    }



In [17]:

# =========================================================
# STEP 1
# TRAIN INITIAL 4-LABEL MODEL
# =========================================================

print("\n================ TRAIN 4 LABEL MODEL ================\n")

model4 = MetaClassifier(
    num_labels=4
)

model4 = train_model(
    model=model4,
    train_loader=train_loader4,
    test_loader=test_loader4,
    epochs=100,
    lr=1e-3
)

torch.save(
    model4.state_dict(),
    "model4.pth"
)

metrics4 = evaluate_model(
    model4,
    test_loader4
)

print("\n4 LABEL RESULTS")
print(metrics4)


# =========================================================
# STEP 2
# EXPAND 4 -> 5
# =========================================================

print("\n================ EXPAND 4 -> 5 ================\n")

model5 = expand_classifier(
    model4,
    new_num_labels=5
)

# ================= PHASE 1 =================
freeze_backbone(model5)

model5 = train_model(
    model=model5,
    train_loader=train_loader5,
    test_loader=test_loader5,
    epochs=100,
    lr=1e-3
)

# ================= PHASE 2 =================
unfreeze_all(model5)

model5 = train_model(
    model=model5,
    train_loader=train_loader5,
    test_loader=test_loader5,
    epochs=10,
    lr=1e-5
)

torch.save(
    model5.state_dict(),
    "model5.pth"
)

metrics5 = evaluate_model(
    model5,
    test_loader5
)

print("\n5 LABEL RESULTS")
print(metrics5)


# =========================================================
# STEP 3
# EXPAND 5 -> 6
# =========================================================

print("\n================ EXPAND 5 -> 6 ================\n")

model6 = expand_classifier(
    model5,
    new_num_labels=6
)

freeze_backbone(model6)

model6 = train_model(
    model=model6,
    train_loader=train_loader6,
    test_loader=test_loader6,
    epochs=100,
    lr=1e-3
)

unfreeze_all(model6)

model6 = train_model(
    model=model6,
    train_loader=train_loader6,
    test_loader=test_loader6,
    epochs=10,
    lr=1e-5
)

torch.save(
    model6.state_dict(),
    "model6.pth"
)

metrics6 = evaluate_model(
    model6,
    test_loader6
)

print("\n6 LABEL RESULTS")
print(metrics6)


# =========================================================
# STEP 4
# EXPAND 6 -> 7
# =========================================================

print("\n================ EXPAND 6 -> 7 ================\n")

model7 = expand_classifier(
    model6,
    new_num_labels=7
)

freeze_backbone(model7)

model7 = train_model(
    model=model7,
    train_loader=train_loader7,
    test_loader=test_loader7,
    epochs=100,
    lr=1e-3
)

unfreeze_all(model7)

model7 = train_model(
    model=model7,
    train_loader=train_loader7,
    test_loader=test_loader7,
    epochs=10,
    lr=1e-5
)

torch.save(
    model7.state_dict(),
    "model7.pth"
)

metrics7 = evaluate_model(
    model7,
    test_loader7
)

print("\n7 LABEL RESULTS")
print(metrics7)


# =========================================================
# STEP 5
# EXPAND 7 -> 8
# =========================================================

print("\n================ EXPAND 7 -> 8 ================\n")

model8 = expand_classifier(
    model7,
    new_num_labels=8
)

freeze_backbone(model8)

model8 = train_model(
    model=model8,
    train_loader=train_loader8,
    test_loader=test_loader8,
    epochs=100,
    lr=1e-3
)

unfreeze_all(model8)

model8 = train_model(
    model=model8,
    train_loader=train_loader8,
    test_loader=test_loader8,
    epochs=10,
    lr=1e-5
)

torch.save(
    model8.state_dict(),
    "model8.pth"
)

metrics8 = evaluate_model(
    model8,
    test_loader8
)

print("\n8 LABEL RESULTS")
print(metrics8)


# =========================================================
# FINAL SUMMARY
# =========================================================

print("\n================ FINAL RESULTS ================\n")

print("4 LABEL:", metrics4)
print("5 LABEL:", metrics5)
print("6 LABEL:", metrics6)
print("7 LABEL:", metrics7)
print("8 LABEL:", metrics8)


================ TRAIN 4 LABEL MODEL ================


4 LABEL RESULTS
{'f1_micro': 0.9344226098476485, 'f1_macro': 0.8626543250350455, 'f1_weighted': 0.9331738763675814, 'f1_samples': 0.8698219507101087, 'precision_micro': 0.9346289752650176, 'precision_macro': 0.8847056769753452, 'precision_weighted': 0.9329811292923493, 'precision_samples': 0.8759137426900584, 'recall_micro': 0.9342163355408388, 'recall_macro': 0.8457669510830695, 'recall_weighted': 0.9342163355408388, 'recall_samples': 0.8767817982456141}

================ EXPAND 4 -> 5 ================


5 LABEL RESULTS
{'f1_micro': 0.9130919997210016, 'f1_macro': 0.7940400501976675, 'f1_weighted': 0.9047429747281945, 'f1_samples': 0.8586933479532163, 'precision_micro': 0.9306839186691312, 'precision_macro': 0.8734569595024798, 'precision_weighted': 0.9257422948220372, 'precision_samples': 0.8737847222222221, 'recall_micro': 0.8961527929901424, 'recall_macro': 0.7525348323713351, 'recall_weighted': 0.8961527929901424, 'recall_sa

In [ ]:
import copy
import torch
import torch.nn as nn

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    hamming_loss 
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# =========================================================
# MODEL
# =========================================================

class MetaClassifier(nn.Module):

    def __init__(self, num_labels=4):

        super().__init__()

        # ================= FEATURE EXTRACTOR =================

        self.feature_extractor = nn.Sequential(

            # ===== Layer 1 =====
            nn.Linear(192, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            # ===== Layer 2 =====
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        # ================= CLASSIFIER =================

        self.classifier = nn.Linear(
            64,
            num_labels
        )

    def forward(self, x):

        x = self.feature_extractor(x)

        logits = self.classifier(x)

        return logits


# =========================================================
# EXPAND CLASSIFIER
# =========================================================

def expand_classifier(
        old_model,
        new_num_labels
):

    old_num_labels = \
        old_model.classifier.out_features

    # ================= CREATE NEW MODEL =================

    new_model = MetaClassifier(
        num_labels=new_num_labels
    )

    # ================= COPY BACKBONE =================

    new_model.feature_extractor.load_state_dict(
        old_model.feature_extractor.state_dict()
    )

    # ================= COPY OLD HEAD =================

    with torch.no_grad():

        new_model.classifier.weight[:old_num_labels] = \
            old_model.classifier.weight

        new_model.classifier.bias[:old_num_labels] = \
            old_model.classifier.bias

    return new_model


# =========================================================
# FREEZE BACKBONE
# =========================================================

def freeze_backbone(model):

    for param in model.feature_extractor.parameters():

        param.requires_grad = False


# =========================================================
# UNFREEZE ALL
# =========================================================

def unfreeze_all(model):

    for param in model.parameters():

        param.requires_grad = True


# =========================================================
# EVALUATION
# =========================================================

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)

import torch


def evaluate_model(
        model,
        data_loader,
        device='cuda',
        threshold=0.5
):

    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():

        for X_batch, y_batch in data_loader:

            X_batch = X_batch.to(device)

            logits = model(X_batch)

            probs = torch.sigmoid(logits)

            preds = (probs > threshold).float()

            y_true.extend(
                y_batch.cpu().numpy()
            )

            y_pred.extend(
                preds.cpu().numpy()
            )

    # =====================================================
    # F1 SCORE
    # =====================================================

    f1_micro = f1_score(
        y_true,
        y_pred,
        average='micro',
        zero_division=0
    )

    f1_macro = f1_score(
        y_true,
        y_pred,
        average='macro',
        zero_division=0
    )

    f1_weighted = f1_score(
        y_true,
        y_pred,
        average='weighted',
        zero_division=0
    )

    f1_samples = f1_score(
        y_true,
        y_pred,
        average='samples',
        zero_division=0
    )

    # =====================================================
    # PRECISION
    # =====================================================

    precision_micro = precision_score(
        y_true,
        y_pred,
        average='micro',
        zero_division=0
    )

    precision_macro = precision_score(
        y_true,
        y_pred,
        average='macro',
        zero_division=0
    )

    precision_weighted = precision_score(
        y_true,
        y_pred,
        average='weighted',
        zero_division=0
    )

    precision_samples = precision_score(
        y_true,
        y_pred,
        average='samples',
        zero_division=0
    )

    # =====================================================
    # RECALL
    # =====================================================

    recall_micro = recall_score(
        y_true,
        y_pred,
        average='micro',
        zero_division=0
    )

    recall_macro = recall_score(
        y_true,
        y_pred,
        average='macro',
        zero_division=0
    )

    recall_weighted = recall_score(
        y_true,
        y_pred,
        average='weighted',
        zero_division=0
    )

    recall_samples = recall_score(
        y_true,
        y_pred,
        average='samples',
        zero_division=0
    )

    # =====================================================
    # RETURN
    # =====================================================

    return {

        # ================= F1 =================
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "f1_samples": f1_samples,

        # ================= PRECISION =================
        "precision_micro": precision_micro,
        "precision_macro": precision_macro,
        "precision_weighted": precision_weighted,
        "precision_samples": precision_samples,

        # ================= RECALL =================
        "recall_micro": recall_micro,
        "recall_macro": recall_macro,
        "recall_weighted": recall_weighted,
        "recall_samples": recall_samples,
    }
# =========================================================
# TRAIN
# =========================================================

def train_model(
        model,
        train_loader,
        test_loader,
        epochs=100,
        lr=1e-3
):

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(
        filter(
            lambda p: p.requires_grad,
            model.parameters()
        ),
        lr=lr
    )

    model.to(device)

    best_model = None

    best_f1 = 0

    for epoch in range(epochs):

        # ================= TRAIN =================

        model.train()

        total_loss = 0

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)

            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            logits = model(X_batch)

            loss = criterion(
                logits,
                y_batch
            )

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        # ================= EVALUATE =================

        metrics = evaluate_model(
            model,
            test_loader
        )

        # print(
        #     f"Epoch [{epoch+1}/{epochs}] | "
        #     f"Loss: {avg_loss:.4f} | "
        #     f"F1-Micro: {metrics['f1_micro']:.4f} | "
        #     f"F1-Macro: {metrics['f1_macro']:.4f}"
        # )

        if metrics['f1_micro'] > best_f1:

            best_f1 = metrics['f1_micro']

            best_model = copy.deepcopy(model)

    return best_model



Device: cuda


In [19]:

# =========================================================
# TRAIN BASE MODEL (4 LABELS)
# =========================================================

print("\n================ BASE MODEL 4 LABELS ================\n")

base_model4 = MetaClassifier(
    num_labels=4
)

base_model4 = train_model(
    model=base_model4,
    train_loader=train_loader4,
    test_loader=test_loader4,
    epochs=100,
    lr=1e-3
)

torch.save(
    base_model4.state_dict(),
    "base_model4.pth"
)

metrics4 = evaluate_model(
    base_model4,
    test_loader4
)

print("\n4 LABEL RESULTS")
print(metrics4)


# =========================================================
# FUNCTION:
# INDEPENDENT ADAPTATION
# =========================================================

def run_adaptation_experiment(
        num_labels,
        train_loader,
        test_loader
):

    print(
        f"\n================ 4 -> {num_labels} ================\n"
    )

    # =====================================================
    # LOAD BASE MODEL
    # =====================================================

    base_model = MetaClassifier(
        num_labels=4
    )

    base_model.load_state_dict(
        torch.load("base_model4.pth")
    )

    # =====================================================
    # EXPAND
    # =====================================================

    adapted_model = expand_classifier(
        base_model,
        new_num_labels=num_labels
    )

    # =====================================================
    # PHASE 1
    # Freeze backbone
    # =====================================================

    print("\nPHASE 1: TRAIN NEW HEAD\n")

    freeze_backbone(adapted_model)

    adapted_model = train_model(
        model=adapted_model,
        train_loader=train_loader,
        test_loader=test_loader,
        epochs=100,
        lr=1e-3
    )

    # =====================================================
    # PHASE 2
    # Fine-tune all
    # =====================================================

    print("\nPHASE 2: FINE-TUNE ALL\n")

    unfreeze_all(adapted_model)

    adapted_model = train_model(
        model=adapted_model,
        train_loader=train_loader,
        test_loader=test_loader,
        epochs=10,
        lr=1e-5
    )

    # =====================================================
    # SAVE
    # =====================================================

    torch.save(
        adapted_model.state_dict(),
        f"model_{num_labels}.pth"
    )

    # =====================================================
    # EVALUATE
    # =====================================================

    metrics = evaluate_model(
        adapted_model,
        test_loader
    )

    print(
        f"\nFINAL RESULTS ({num_labels} LABELS)"
    )

    print(metrics)

    return metrics


# =========================================================
# 4 -> 5
# =========================================================

metrics5 = run_adaptation_experiment(
    num_labels=5,
    train_loader=train_loader5,
    test_loader=test_loader5
)


# =========================================================
# 4 -> 6
# =========================================================

metrics6 = run_adaptation_experiment(
    num_labels=6,
    train_loader=train_loader6,
    test_loader=test_loader6
)


# =========================================================
# 4 -> 7
# =========================================================

metrics7 = run_adaptation_experiment(
    num_labels=7,
    train_loader=train_loader7,
    test_loader=test_loader7
)


# =========================================================
# 4 -> 8
# =========================================================

metrics8 = run_adaptation_experiment(
    num_labels=8,
    train_loader=train_loader8,
    test_loader=test_loader8
)


# =========================================================
# FINAL SUMMARY
# =========================================================

print("\n================ FINAL SUMMARY ================\n")

print("\n4 LABELS")
print(metrics4)

print("\n4 -> 5")
print(metrics5)

print("\n4 -> 6")
print(metrics6)

print("\n4 -> 7")
print(metrics7)

print("\n4 -> 8")
print(metrics8)


================ BASE MODEL 4 LABELS ================


4 LABEL RESULTS
{'f1_micro': 0.9340986863892262, 'f1_macro': 0.8608556457081762, 'f1_weighted': 0.9326729163445112, 'f1_samples': 0.8693191311612364, 'precision_micro': 0.9342018105542063, 'precision_macro': 0.8865076674826203, 'precision_weighted': 0.9325268908781208, 'precision_samples': 0.8753198099415205, 'recall_micro': 0.9339955849889625, 'recall_macro': 0.8430926289006383, 'recall_weighted': 0.9339955849889625, 'recall_samples': 0.8766538742690059}

================ 4 -> 5 ================


PHASE 1: TRAIN NEW HEAD



C:\Users\Admin\AppData\Local\Temp\ipykernel_15096\223647352.py:57: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("base_model4.pth")



PHASE 2: FINE-TUNE ALL


FINAL RESULTS (5 LABELS)
{'f1_micro': 0.9103722569165823, 'f1_macro': 0.7813015572249887, 'f1_weighted': 0.8996952741655346, 'f1_samples': 0.8587506961849067, 'precision_micro': 0.9282817502668089, 'precision_macro': 0.8700004000432431, 'precision_weighted': 0.921995026759436, 'precision_samples': 0.8755902777777778, 'recall_micro': 0.8931407447973713, 'recall_macro': 0.7403918395457695, 'recall_weighted': 0.8931407447973713, 'recall_samples': 0.8592726608187135}

================ 4 -> 6 ================


PHASE 1: TRAIN NEW HEAD



C:\Users\Admin\AppData\Local\Temp\ipykernel_15096\223647352.py:57: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("base_model4.pth")



PHASE 2: FINE-TUNE ALL


FINAL RESULTS (6 LABELS)
{'f1_micro': 0.8298228817616085, 'f1_macro': 0.7082621192830209, 'f1_weighted': 0.7852004887930668, 'f1_samples': 0.7658231990962253, 'precision_micro': 0.9234252230656546, 'precision_macro': 0.8642212268975501, 'precision_weighted': 0.9089771264017521, 'precision_samples': 0.8718823099415206, 'recall_micro': 0.7534499619689232, 'recall_macro': 0.6533261377614171, 'recall_weighted': 0.7534499619689232, 'recall_samples': 0.7219919590643273}

================ 4 -> 7 ================


PHASE 1: TRAIN NEW HEAD



C:\Users\Admin\AppData\Local\Temp\ipykernel_15096\223647352.py:57: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("base_model4.pth")



PHASE 2: FINE-TUNE ALL


FINAL RESULTS (7 LABELS)
{'f1_micro': 0.8170832839710934, 'f1_macro': 0.6214798690142864, 'f1_weighted': 0.7645818933219336, 'f1_samples': 0.757310051060051, 'precision_micro': 0.9257718120805369, 'precision_macro': 0.8786582738583341, 'precision_weighted': 0.910526773458792, 'precision_samples': 0.8709463763575607, 'recall_micro': 0.7312340966921119, 'recall_macro': 0.5634518236599881, 'recall_weighted': 0.7312340966921119, 'recall_samples': 0.7082189849624061}

================ 4 -> 8 ================


PHASE 1: TRAIN NEW HEAD



C:\Users\Admin\AppData\Local\Temp\ipykernel_15096\223647352.py:57: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("base_model4.pth")



PHASE 2: FINE-TUNE ALL


FINAL RESULTS (8 LABELS)
{'f1_micro': 0.8273953985049995, 'f1_macro': 0.6430385611789566, 'f1_weighted': 0.7811883267545658, 'f1_samples': 0.7782293230648494, 'precision_micro': 0.9314245123217311, 'precision_macro': 0.876914879336216, 'precision_weighted': 0.9175568969150014, 'precision_samples': 0.8713345864661654, 'recall_micro': 0.7442693096974196, 'recall_macro': 0.585571480937978, 'recall_weighted': 0.7442693096974196, 'recall_samples': 0.731467209690894}

================ FINAL SUMMARY ================


4 LABELS
{'f1_micro': 0.9340986863892262, 'f1_macro': 0.8608556457081762, 'f1_weighted': 0.9326729163445112, 'f1_samples': 0.8693191311612364, 'precision_micro': 0.9342018105542063, 'precision_macro': 0.8865076674826203, 'precision_weighted': 0.9325268908781208, 'precision_samples': 0.8753198099415205, 'recall_micro': 0.9339955849889625, 'recall_macro': 0.8430926289006383, 'recall_weighted': 0.9339955849889625, 'recall_samples': 0.8766538742690059}

4 -